In [1]:
import pandas as pd
import re
from collections import Counter

# 读取Excel文件,数据太大，先对2020年数据测试，
df1 = pd.read_excel('../data/膝关节MR2020.xls')
# df2 = pd.read_excel('file2.xlsx')

# 获取内容
data = []
if '征象描述' in df1.columns:
    data += df1['征象描述'].tolist()
# if '征象描述' in df2.columns:
#     data += df2['征象描述'].tolist()

# 将文本分割为句子
sentences = []
for d in data:
    # 如果文本中包含“术后”两个字，则删除“术后”之后的内容
    if '术后' in str(d):
#         d = re.split('术后', str(d))[0] + '术后'
        d = re.split('术后', str(d))[0]
    s = re.split('[,，：:。.；?？\n]', str(d))
    s = [i.strip() for i in s if i.strip() != '']
    sentences.extend(s)

# 将每一个断句以换行为间隔符保存在output.txt文件中
with open('../output/separate_output.txt', 'w', encoding='utf-8') as f:
    for s in sentences:
        f.write(s + '\n')

print('Done!')


Done!


In [3]:
# 读取文件中的内容
with open('../output/separate_output.txt', 'r', encoding='utf-8') as f:
    content = f.read()

# 将文本分割为字段
fields = content.strip().split('\n')

# 统计每个字段的出现次数
counts = Counter(fields)

# 计算每个字段的频率
total = len(fields)
frequencies = {k: v / total for k, v in counts.items()}

# 转换为pandas DataFrame对象
df = pd.DataFrame({'字段内容': list(counts.keys()), '出现次数': list(counts.values()), '频率': list(frequencies.values())})

# 按出现次数排序
df = df.sort_values('出现次数', ascending=False)

# 将结果写入Excel文件
df.to_excel('../output/frequent_output.xlsx', index=False)

print('Done!')
print('total: {}'.format(total))


Done!
total: 107341


In [4]:
import jieba
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import random

# 读取文本数据，随机取10000条进行分析，不然内存会达到15G，报错
with open('../output/separate_output.txt', 'r', encoding='utf-8') as f:
    data = f.readlines()
    data = random.sample(data, 10000)

# 构建TF-IDF矩阵
vectorizer = TfidfVectorizer(max_df=1.0, min_df=1)
X = vectorizer.fit_transform(data)

# TfidfVectorizer向量化
tfidf = TfidfVectorizer()
X = tfidf.fit_transform(data)

# 聚类
kmeans = KMeans(n_clusters=10, random_state=0)
kmeans.fit(X)

D:\IT\python\pythonEnvironment\envs\nlpHelper\lib\site-packages\sklearn\cluster\_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


KMeans(n_clusters=10, random_state=0)

In [5]:
# 将聚类结果写入文件
with open('../output/KMeans_output.txt', 'w', encoding='utf-8') as f:
    for i in range(10):
        f.write(f'Cluster {i}:\n')
        for j, sentence in enumerate(data):
            if kmeans.labels_[j] == i:
                f.write(sentence)
        f.write('\n')
print('Done!')

Done!


In [6]:
# 使用轮廓系数评估聚类结果
from sklearn.metrics import silhouette_score
score = silhouette_score(X, kmeans.labels_, metric='euclidean')
print(f"轮廓系数为: {score}")

轮廓系数为: 0.19208718600946495
